# Multifactorial Meta-Learning Benchmark for Small-Data Machine Learning in Soft Matter and Biomaterials Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR⁽²⁾ dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.0bxf-db06/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` (as per Croissant specification).


In [ ]:
# List all available record sets with their @id
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        print(f"RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
        record_sets.append(rs['@id'])
else:
    print("No record sets found in this dataset metadata.")

# If RecordSet details available, show their fields
fields_by_record = {}
for rs_id in record_sets:
    rs = None
    # Try to fetch the full RecordSet object
    for r in metadata.recordSet:
        if r['@id'] == rs_id:
            rs = r
            break
    if rs and 'field' in rs:
        fields = rs['field']
        print(f"Fields for RecordSet {rs_id}:")
        for f in fields:
            print(f"  Field @id: {f['@id']} Name: {f.get('name', 'N/A')} DataType: {f.get('dataType', 'N/A')}")
        fields_by_record[rs_id] = [f['@id'] for f in fields]
    else:
        print(f"No fields found for RecordSet {rs_id}.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
If there are multiple record sets, we loop through each and load their records.

In [ ]:
dataframes = {}

# If record sets exist, extract records from each
for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields (@id) in DataFrame: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for RecordSet {record_set_id}.")

if not dataframes:
    print("No dataframes were loaded from records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Use `@id` values to reference fields.
--
**If you wish to customize the next cells, choose one of the loaded record sets and one of its numeric field `@id`s.**

In [ ]:
# Example: Choose a record set and numeric field for analysis
if dataframes:
    # Pick the first dataframe loaded
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id: {record_set_id}")

    # Try to identify a numeric field (@id) from its datatype (float/int)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Try grouping by another field (e.g., a categorical field)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found in DataFrame for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Data is referenced by `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram or scatter plot for the numeric field and group field if they exist
if dataframes and 'numeric_field_id' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[record_set_id][numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[record_set_id])
        plt.title(f'{numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


*The FAIR⁽²⁾ dataset provides a benchmark suite for meta-learning and reproducibility in soft matter and biomaterials. Using `mlcroissant`, we can efficiently load, explore, and process its contents, referencing all entities by their unique `@id`. Further analysis can be performed on individual record sets and fields, including statistical summaries and advanced visualizations for empirical machine learning workflows.*